# ch05 Python 附录：FWL、固定效应与 DID

连享会 2026 暑期班·初级班 · 第 5 讲的 Python 对照。与 `lectures/05_FE_intFE.qmd` 代码实操一一对应，用 Python 复现辛普森悖论、个体/时间/双向固定效应与 2×2 DID。

**呈现约定**：本笔记本为静态展示，输出为本机运行后冻结；正文以 Stata 为主线，Python 作对照。

**环境**：`pandas`、`numpy`、`statsmodels`；固定效应用 `linearmodels`（`PanelOLS`）或 `pyfixest`（`feols`）。

```
pip install pandas numpy statsmodels linearmodels
```

## 任务说明与提示词

把本讲 `examples/ch05/ch05_main.do` 中的关键 Stata 代码翻成等价 Python。可用如下提示词让 AI 生成/校对（对应 skill：`core/05-regression-interpreter`）：

> 下面这段 Stata 用双向固定效应估计 `y` 对 `x` 的关系：`reghdfe y x, absorb(id year) cluster(id)`。请用 Python 的 `linearmodels.PanelOLS`（或 `pyfixest`）写出等价实现，含设定面板索引、加入个体与时间固定效应、按 `id` 聚类稳健标准误，并逐行加中文注释说明它与 Stata 命令的对应关系。只写等价实现，不改变模型设定。

## 1. 辛普森悖论与组内去心

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(20260706)
# 三家公司：个体效应与初始规模负相关，真实 beta = 0.5
rows = []
for i in range(1, 4):
    a_i, size0 = 12 - 3 * i, 2 + 3 * i
    for _ in range(100):
        x = size0 + rng.uniform(0, 4)
        y = a_i + 0.5 * x + rng.normal(0, 0.8)
        rows.append((i, x, y))
df = pd.DataFrame(rows, columns=['id', 'x', 'y'])

# 混合回归：斜率为负（被公司间差异污染）
print('混合回归 :', smf.ols('y ~ x', df).fit().params['x'].round(3))
# 组内去心（个体固定效应）：斜率回到 0.5 附近
print('个体 FE  :', smf.ols('y ~ x + C(id)', df).fit().params['x'].round(3))

## 2. FWL 定理：先剔除，再回归

用「残差对残差」验证 FWL：`y ~ x1 + x2` 中 `x1` 的系数，等于「x1 剔除 x2 的残差」对「y 剔除 x2 的残差」回归的斜率。

In [ ]:
# 沿用上面的 df，把 C(id) 当作 x2（此处用 id 均值近似演示残差化思想）
df['x_res'] = df['x'] - df.groupby('id')['x'].transform('mean')
df['y_res'] = df['y'] - df.groupby('id')['y'].transform('mean')
print('一起回归 y~x+C(id) 的 x 系数 :', smf.ols('y ~ x + C(id)', df).fit().params['x'].round(3))
print('残差对残差 y_res~x_res 的系数 :', smf.ols('y_res ~ x_res', df).fit().params['x_res'].round(3))

## 3. 个体、时间与双向固定效应（linearmodels）

In [ ]:
# 需 pip install linearmodels
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

# nlswork：statsmodels 无自带，此处用模拟面板演示写法（真实数据可从 Stata 导出）
rng = np.random.default_rng(2026)
N, T = 200, 6
panel = []
for i in range(N):
    ai = rng.normal(0, 1)
    for t in range(T):
        x = rng.normal(0, 1)
        y = ai + 0.4 * x + 0.1 * t + rng.normal(0, 0.5)
        panel.append((i, 2010 + t, x, y))
pdata = pd.DataFrame(panel, columns=['id', 'year', 'x', 'y']).set_index(['id', 'year'])

# 个体 FE
m_id = PanelOLS(pdata['y'], sm.add_constant(pdata[['x']]), entity_effects=True)
# 双向 FE（个体 + 时间），按个体聚类稳健
m_tw = PanelOLS(pdata['y'], sm.add_constant(pdata[['x']]), entity_effects=True, time_effects=True)
print('个体 FE  x 系数 :', round(m_id.fit(cov_type='clustered', cluster_entity=True).params['x'], 3))
print('双向 FE  x 系数 :', round(m_tw.fit(cov_type='clustered', cluster_entity=True).params['x'], 3))

## 4. 2×2 DID：还原政策效应

真实政策效应设为 +3，看交乘项 `treat:post` 能否还原。

In [ ]:
rng = np.random.default_rng(20260708)
rows = []
for i in range(200):
    treat = 1 if i >= 100 else 0
    for post in (0, 1):
        y = 10 + 2 * treat + 4 * post + 3 * (treat * post) + rng.normal(0, 1)
        rows.append((i, treat, post, y))
did = pd.DataFrame(rows, columns=['id', 'treat', 'post', 'y'])

# 四格均值
print(did.groupby(['post', 'treat'])['y'].mean().round(2))
# 交乘项 treat:post 应还原 3
res = smf.ols('y ~ treat * post', did).fit(cov_type='HC1')
print(res.params.round(2))

---
完整概念与解读见 `lectures/05_FE_intFE.qmd`。现代 DID 估计量（CSDID 等）与 DDML 留高级班：<https://www.lianxh.cn/details/1811.html>